# Retrain SATD Detector: 100% Sesuai Metodologi Eksperimen Jurnal
**Paper:** *Automatic identification of self-admitted technical debt from four different sources* (Li et al., EMSE 2023)
**Official Repository:** [yikun-li/satd-different-sources-data](https://github.com/yikun-li/satd-different-sources-data)

---
### Metodologi Eksperimen Resmi Paper yang Diterapkan Penuh:
1. **Arsitektur Model Pendekatan Penulis:** `TextCNNMultitask` (kernel `[1, 2, 3, 4, 5]`, 200 filters = 1000 feature maps, dropout 0.5, 4 task heads) sesuai `satd_detector.py`.
2. **Stratified 10-Fold Cross-Validation (Bab 3.6.4):** Evaluasi 10 putaran berstrata di ke-4 dataset sumber data (`code_comment`, `commit_message`, `pull_request`, `issue`).
3. **Weighted CrossEntropyLoss (Bab 3.6.6):** Mengatasi *class imbalance* dengan bobot frekuensi terbalik (*inverse class frequency weights*) per task agar kelas minoritas (seperti REQ debt pada commit) tidak bernilai 0.
4. **Stochastic Multitask Training (Bab 3.6.5 - Collobert & Weston 2008):** Setiap langkah iterasi mengambil mini-batch secara acak bergantian dari 4 sumber artefak.
5. **Metrik & Format Evaluasi:** Precision, Recall, F1 per tipe debt (`C/D.`, `REQ.`, `DOC.`, `TST.`), Macro F1 (`AVG.`), serta rata-rata 10-Fold.
6. **Export Model & Inferensi:** Menyimpan checkpoint model terbaik ke `satd_detector.pt` dan vocabulary ke `satd_vocab.json`, serta pengujian kalimat langsung (`simple_test`).

## Cell 1: Environment & GPU Setup

In [ ]:
import os
import re
import json
import string
import random
import warnings
import numpy as np
import pandas as pd
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_fscore_support

import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_gpus = torch.cuda.device_count()
gpu_names = [torch.cuda.get_device_name(i) for i in range(num_gpus)] if num_gpus > 0 else []
print(f'Device: {device} | Total GPU: {num_gpus} {gpu_names}')

# Parameter Eksperimen Sesuai Paper (Li et al. 2023)
N_SPLITS = 10       # Stratified 10-Fold CV (Bab 3.6.4)
EPOCHS = 10         # 10 Epoch per fold (Kim 2014 & Li et al.)
BATCH_SIZE = 64     # Ukuran mini-batch
LEARNING_RATE = 1e-3
EMBED_DIM = 300     # Dimensi embedding
KERNEL_NUM = 200    # 200 feature maps per kernel (satd_detector.py)
KERNEL_SIZES = [1, 2, 3, 4, 5]  # Multi-window kernels (satd_detector.py)
DROPOUT = 0.5
MAX_LEN = 100

# Opsi verifikasi cepat:
# Set QUICK_TEST = True jika ingin menjalankan 2 fold awal untuk pengujian pipeline kilat.
# Set QUICK_TEST = False untuk menjalankan full 10-Fold CV eksperimen resmi jurnal.
QUICK_TEST = False
if QUICK_TEST:
    N_SPLITS = 2
    print(f'MODE CEPAT DIAKTIFKAN: Menjalankan {N_SPLITS} Fold')
else:
    print(f'MODE FULL EKSPERIMEN JURNAL: Menjalankan {N_SPLITS} Fold')

## Cell 2: Auto-Download Dataset dari GitHub Repo Yikun Li

In [ ]:
GITHUB_REPO_URL = 'https://github.com/yikun-li/satd-different-sources-data.git'
RAW_BASE_URL = 'https://raw.githubusercontent.com/yikun-li/satd-different-sources-data/main'
DATA_DIR = './data'

required_files = [
    'satd-dataset-code_comments.csv',
    'satd-dataset-commit_messages.csv',
    'satd-dataset-pull_requests.csv',
    'satd-dataset-issues.csv'
]

os.makedirs(DATA_DIR, exist_ok=True)
all_present = all(os.path.exists(os.path.join(DATA_DIR, f)) for f in required_files)

if not all_present:
    print('Mengunduh dataset dari repositori GitHub Yikun Li...')
    exit_code = os.system(f'git clone {GITHUB_REPO_URL} /tmp/satd_repo && cp /tmp/satd_repo/*.csv {DATA_DIR}/ && rm -rf /tmp/satd_repo')
    if exit_code != 0 or not all(os.path.exists(os.path.join(DATA_DIR, f)) for f in required_files):
        for f in required_files:
            dest = os.path.join(DATA_DIR, f)
            if not os.path.exists(dest):
                url = f'{RAW_BASE_URL}/{f}'
                print(f'Mengunduh {f}...')
                urllib.request.urlretrieve(url, dest)
    print('Seluruh file dataset berhasil diunduh ke folder:', DATA_DIR)
else:
    print('Dataset sudah lengkap di folder:', DATA_DIR)

## Cell 3: Preprocessing & Label Mapping Sesuai `satd_detector.py`

In [ ]:
DEF_ISSUE = 'issue'
DEF_COMMIT = 'commit_message'
DEF_COMMENT = 'code_comment'
DEF_PULL = 'pull_request'

DEF_MAPPING = {
    DEF_ISSUE: 0,
    DEF_COMMIT: 1,
    DEF_COMMENT: 2,
    DEF_PULL: 3
}

DEF_LABELS = [
    'non-SATD',            # Index 0
    'code|design-debt',    # Index 1 (C/D.)
    'requirement-debt',    # Index 2 (REQ.)
    'documentation-debt',  # Index 3 (DOC.)
    'test-debt'            # Index 4 (TST.)
]

DATA_FILES = {
    DEF_COMMENT: os.path.join(DATA_DIR, 'satd-dataset-code_comments.csv'),
    DEF_COMMIT: os.path.join(DATA_DIR, 'satd-dataset-commit_messages.csv'),
    DEF_PULL: os.path.join(DATA_DIR, 'satd-dataset-pull_requests.csv'),
    DEF_ISSUE: os.path.join(DATA_DIR, 'satd-dataset-issues.csv')
}

def map_raw_label(raw_label):
    raw = str(raw_label).lower().strip()
    if 'code' in raw or 'design' in raw:
        return 1  # code|design-debt
    elif 'requirement' in raw:
        return 2  # requirement-debt
    elif 'documentation' in raw:
        return 3  # documentation-debt
    elif 'test' in raw:
        return 4  # test-debt
    else:
        return 0  # non-SATD

tokenizer_words = nltk.TweetTokenizer()
punctuation_chars = string.punctuation.replace('!', '').replace('?', '')
padding_token = '<pad>'

def comment_pre_processing(comment):
    if not isinstance(comment, str) or not comment.strip():
        return ['<pad>'] * 5
    comment = re.sub(r'(//)|(/\*)|(\*/)', '', comment).lower()
    try:
        tokens_sentences = [tokenizer_words.tokenize(t) for t in nltk.sent_tokenize(comment)]
    except Exception:
        tokens_sentences = [tokenizer_words.tokenize(comment)]

    processed_tokens_sentences = []
    for sentence in tokens_sentences:
        processed_tokens = []
        for token in sentence:
            if token == 'non-nls' or token == padding_token:
                processed_tokens.append(token)
                continue
            elif token == ',':
                processed_tokens.append('.')
            elif any(c in token for c in [' ', '.', '#', '_', '/', '>', '<', '@']):
                continue
            elif any(c.isdigit() for c in token):
                continue
            else:
                processed_tokens.append(token)

        if len(processed_tokens) > 0 and processed_tokens[-1] != '.':
            processed_tokens.append('.')
        processed_tokens_sentences.append(processed_tokens)

    tokens = [w for t in processed_tokens_sentences for w in t]
    stripped = [w for w in tokens if w and (w not in punctuation_chars and ':' not in w and '=' not in w and ')' not in w and '(' not in w)]
    if len(stripped) < 5:
        stripped = stripped + [padding_token] * (5 - len(stripped))
    return stripped

dfs = {}
print('Memuat dan membersihkan seluruh dataset teks...')
for tp, file_path in DATA_FILES.items():
    df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip')
    text_col = 'text' if 'text' in df.columns else df.columns[1]
    cls_col = 'classification' if 'classification' in df.columns else ('label' if 'label' in df.columns else df.columns[-1])
    
    df['clean_label'] = df[cls_col].apply(map_raw_label)
    df['tokens'] = df[text_col].astype(str).apply(comment_pre_processing)
    dfs[tp] = df[['tokens', 'clean_label']].copy()
    counts = df['clean_label'].value_counts().to_dict()
    print(f'  {tp:<15}: {len(df)} sampel | Dist: non-SATD={counts.get(0, 0)}, C/D={counts.get(1, 0)}, REQ={counts.get(2, 0)}, DOC={counts.get(3, 0)}, TST={counts.get(4, 0)}')

## Cell 4: Setup Vocabulary, Stratified 10-Fold CV & Weighted Loss Helper

In [ ]:
class SATDVocabulary:
    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.word2idx = {'<pad>': 0, '<unk>': 1}
        self.idx2word = {0: '<pad>', 1: '<unk>'}
        self.counts = {}

    def build(self, token_lists):
        for toks in token_lists:
            for tok in toks:
                self.counts[tok] = self.counts.get(tok, 0) + 1
        idx = 2
        for tok, cnt in self.counts.items():
            if cnt >= self.min_freq and tok not in self.word2idx:
                self.word2idx[tok] = idx
                self.idx2word[idx] = tok
                idx += 1

    def numericalize(self, tokens, max_len=MAX_LEN):
        indices = [self.word2idx.get(t, 1) for t in tokens[:max_len]]
        if len(indices) < max_len:
            indices += [0] * (max_len - len(indices))
        return indices

    def __len__(self):
        return len(self.word2idx)

# Bangun vocabulary korpus
all_tokens = []
for tp in dfs:
    all_tokens.extend(dfs[tp]['tokens'].tolist())

vocab = SATDVocabulary(min_freq=2)
vocab.build(all_tokens)
print(f'Total Vocabulary Tokens: {len(vocab)}')

class SATDTorchDataset(Dataset):
    def __init__(self, token_list, labels, vocab, max_len=MAX_LEN):
        self.features = [vocab.numericalize(toks, max_len) for toks in token_list]
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return torch.tensor(self.features[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

# Pembagian Stratified 10-Fold CV (Bab 3.6.4)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
kfold_splits = {}
for tp in dfs:
    X = dfs[tp]['tokens'].values
    y = dfs[tp]['clean_label'].values
    kfold_splits[tp] = list(skf.split(X, y))

print(f'Partisi Stratified {N_SPLITS}-Fold siap untuk 4 sumber data.')

# Helper: Menghitung Inverse Class Weights per Task (Bab 3.6.6)
def compute_task_loss_weights(y_train, num_classes=5, max_weight=50.0):
    counts = np.bincount(y_train, minlength=num_classes)
    total = len(y_train)
    weights = np.zeros(num_classes, dtype=np.float32)
    for c in range(num_classes):
        if counts[c] > 0:
            weights[c] = total / (num_classes * counts[c])
        else:
            weights[c] = 1.0
    weights = np.clip(weights, 0.1, max_weight)
    return torch.tensor(weights, dtype=torch.float).to(device)

# Helper: Menghitung Metrik Evaluasi F1 Sesuai Paper (Bab 3.6.7)
def calculate_f1_metrics(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=[1, 2, 3, 4], zero_division=0)
    return {
        'C/D.': f1[0],
        'REQ.': f1[1],
        'DOC.': f1[2],
        'TST.': f1[3],
        'AVG.': np.mean(f1)
    }

## Cell 5: Arsitektur Model Multitask Text-CNN (Identik dengan `satd_detector.py`)

In [ ]:
class TextCNNMultitaskTrainable(nn.Module):
    """
    Arsitektur TextCNN Multitask resmi Yikun Li (EMSE 2023) dengan trainable embedding.
    Struktur bobot dan layer identik 100% dengan TextCNNMultitask pada satd_detector.py.
    """
    def __init__(self, vocab_size, embed_dim=EMBED_DIM, kernel_num=KERNEL_NUM, kernel_sizes=KERNEL_SIZES, dropout=DROPOUT):
        super(TextCNNMultitaskTrainable, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.list_conv = nn.ModuleList([
            nn.Conv2d(1, kernel_num, (size, embed_dim)) for size in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fcs = nn.ModuleList([
            nn.Linear(len(kernel_sizes) * kernel_num, len(DEF_LABELS)) for _ in range(len(DEF_MAPPING.keys()))
        ])

    def forward(self, x, task_id=None):
        x_embed = self.embed(x).unsqueeze(1)  # (batch_size, 1, seq_len, embed_dim)
        x_list_conv = [F.relu(conv(x_embed)).squeeze(3) for conv in self.list_conv]
        x_list_max_pool = [F.max_pool1d(x_c, x_c.size(2)).squeeze(2) for x_c in x_list_conv]
        x_cat = torch.cat(x_list_max_pool, 1)
        x_drop = self.dropout(x_cat)
        
        if task_id is not None:
            return self.fcs[task_id](x_drop)
        
        return [fc(x_drop) for fc in self.fcs]

# Tes inisialisasi arsitektur
test_net = TextCNNMultitaskTrainable(vocab_size=len(vocab)).to(device)
print(test_net)

## Cell 6: Stratified 10-Fold CV Training Loop dengan Weighted Loss & Stochastic Multitask

In [ ]:
source_tasks = [DEF_ISSUE, DEF_COMMIT, DEF_COMMENT, DEF_PULL]
fold_results = {tp: [] for tp in source_tasks}
best_overall_f1 = -1.0
BEST_MODEL_PATH = 'satd_detector.pt'

print(f'=== MEMULAI RETRAINING MULTITASK TEXT-CNN DENGAN STRATIFIED {N_SPLITS}-FOLD CV & WEIGHTED LOSS ===\n')

for fold in range(N_SPLITS):
    print('=' * 50)
    print(f'>>> FOLD [{fold + 1:02d}/{N_SPLITS:02d}] <<<')
    print('=' * 50)
    
    # Inisialisasi model segar untuk setiap fold agar evaluasi objektif dan bebas leakage
    model = TextCNNMultitaskTrainable(vocab_size=len(vocab)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    train_loaders = {}
    val_loaders = {}
    task_criterions = {}
    
    # Setup DataLoader & Weighted Loss per Task untuk Fold saat ini
    for tp in source_tasks:
        train_idx, test_idx = kfold_splits[tp][fold]
        X_tr = dfs[tp]['tokens'].values[train_idx]
        y_tr = dfs[tp]['clean_label'].values[train_idx]
        X_te = dfs[tp]['tokens'].values[test_idx]
        y_te = dfs[tp]['clean_label'].values[test_idx]
        
        tr_ds = SATDTorchDataset(X_tr, y_tr, vocab)
        te_ds = SATDTorchDataset(X_te, y_te, vocab)
        
        train_loaders[tp] = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loaders[tp] = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False)
        
        # Hitung Weighted CrossEntropyLoss untuk mengatasi class imbalance (Bab 3.6.6)
        weights = compute_task_loss_weights(y_tr)
        task_criterions[tp] = nn.CrossEntropyLoss(weight=weights)
    
    # Stochastic Multitask Training Loop (Collobert & Weston 2008)
    for epoch in range(1, EPOCHS + 1):
        model.train()
        iterators = {tp: iter(train_loaders[tp]) for tp in source_tasks}
        finished = {tp: False for tp in source_tasks}
        
        running_loss = {tp: 0.0 for tp in source_tasks}
        task_steps = {tp: 0 for tp in source_tasks}
        
        while not all(finished.values()):
            avail = [tp for tp in source_tasks if not finished[tp]]
            if not avail:
                break
            
            chosen_task = random.choice(avail)
            task_id = DEF_MAPPING[chosen_task]
            
            try:
                bx, by = next(iterators[chosen_task])
            except StopIteration:
                finished[chosen_task] = True
                continue
                
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            logits = model(bx, task_id=task_id)
            loss = task_criterions[chosen_task](logits, by)
            loss.backward()
            optimizer.step()
            
            running_loss[chosen_task] += loss.item()
            task_steps[chosen_task] += 1
            
        if epoch % 5 == 0 or epoch == EPOCHS:
            total_l = sum(running_loss.values()) / max(sum(task_steps.values()), 1)
            print(f'  Epoch [{epoch:02d}/{EPOCHS:02d}] - Loss Rata-rata: {total_l:.4f}')
    
    # Evaluasi Fold ini pada data uji (10% test)
    model.eval()
    fold_f1_list = []
    print(f'\n  Hasil Evaluasi Fold {fold + 1}:')
    with torch.no_grad():
        for tp in source_tasks:
            task_id = DEF_MAPPING[tp]
            preds, trues = [], []
            for bx, by in val_loaders[tp]:
                bx = bx.to(device)
                out = model(bx, task_id=task_id)
                pred = torch.argmax(out, dim=1).cpu().numpy()
                preds.extend(pred)
                trues.extend(by.numpy())
                
            m = calculate_f1_metrics(trues, preds)
            fold_results[tp].append(m)
            fold_f1_list.append(m['AVG.'])
            print(f'    - {tp:<15}: C/D={m["C/D."]:.3f} | REQ={m["REQ."]:.3f} | DOC={m["DOC."]:.3f} | TST={m["TST."]:.3f} | AVG={m["AVG."]:.3f}')
    
    fold_mean_avg = np.mean(fold_f1_list)
    print(f'  ==> Fold {fold + 1} Overall Macro F1: {fold_mean_avg:.4f}\n')
    
    # Simpan checkpoint fold terbaik ke satd_detector.pt
    if fold_mean_avg > best_overall_f1:
        best_overall_f1 = fold_mean_avg
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        with open('satd_vocab.json', 'w', encoding='utf-8') as f:
            json.dump(vocab.word2idx, f, ensure_ascii=False, indent=2)
        print(f'  [*] Checkpoint terbaik baru disimpan (Fold {fold + 1} F1: {best_overall_f1:.4f}) ke {BEST_MODEL_PATH}')

print('\n' + '=' * 65)
print('SELURUH STRATIFIED 10-FOLD CV SELESAI!')
print('=' * 65)

## Cell 7: Rekapitulasi Rata-Rata 10-Fold CV (Format Standar Tabel Jurnal)

In [ ]:
# Hitung rata-rata 10-Fold untuk setiap task dan debt type
summary_rows = []
task_pretty = {
    DEF_COMMENT: 'Code Comment',
    DEF_COMMIT: 'Commit Message',
    DEF_PULL: 'Pull Request',
    DEF_ISSUE: 'Issue Tracker'
}

for tp in [DEF_COMMENT, DEF_COMMIT, DEF_PULL, DEF_ISSUE]:
    all_fold_m = fold_results[tp]
    c_d = np.mean([fm['C/D.'] for fm in all_fold_m])
    req = np.mean([fm['REQ.'] for fm in all_fold_m])
    doc = np.mean([fm['DOC.'] for fm in all_fold_m])
    tst = np.mean([fm['TST.'] for fm in all_fold_m])
    avg = np.mean([fm['AVG.'] for fm in all_fold_m])
    
    summary_rows.append({
        'Source Task': task_pretty[tp],
        'C/D.': f'{c_d:.3f}',
        'REQ.': f'{req:.3f}',
        'DOC.': f'{doc:.3f}',
        'TST.': f'{tst:.3f}',
        'AVG. (Macro)': f'{avg:.3f}'
    })

df_summary = pd.DataFrame(summary_rows)
print('=' * 70)
print('REKAPITULASI RATA-RATA EVALUASI STRATIFIED 10-FOLD CV MT-TEXT-CNN')
print('=' * 70)
print(df_summary.to_string(index=False))

# Overall Across Sources
overall_cd = np.mean([float(r['C/D.']) for r in summary_rows])
overall_req = np.mean([float(r['REQ.']) for r in summary_rows])
overall_doc = np.mean([float(r['DOC.']) for r in summary_rows])
overall_tst = np.mean([float(r['TST.']) for r in summary_rows])
overall_avg = np.mean([float(r['AVG. (Macro)']) for r in summary_rows])
print('-' * 70)
print(f'RATA-RATA KESELURUHAN LINTAS 4 SUMBER: {overall_avg:.3f}')
print(f'  (C/D: {overall_cd:.3f} | REQ: {overall_req:.3f} | DOC: {overall_doc:.3f} | TST: {overall_tst:.3f})')
print('=' * 70)

## Cell 8: Pengujian Inferensi Langsung Menggunakan Model Checkpoint Terbaik

In [ ]:
# Load model checkpoint terbaik
best_model = TextCNNMultitaskTrainable(vocab_size=len(vocab)).to(device)
best_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
best_model.eval()
print(f'Model terbaik berhasil dimuat dari: {BEST_MODEL_PATH}')

def classify_satd(text, source_type):
    best_model.eval()
    toks = comment_pre_processing(text)
    idx_seq = vocab.numericalize(toks, max_len=MAX_LEN)
    
    inp = torch.tensor([idx_seq], dtype=torch.long).to(device)
    task_id = DEF_MAPPING[source_type]
    
    with torch.no_grad():
        logits = best_model(inp, task_id=task_id)
        pred_idx = torch.argmax(logits, dim=1).item()
        pred_label = DEF_LABELS[pred_idx]
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]
        
    print(f'Source: {source_type}')
    print(f'Text  : "{text}"')
    print(f'Pred  : {pred_label} (Confidence: {probs[pred_idx]:.2%})\n')
    return pred_label

print('=== TEST INFERENSI DENGAN SUITE RESMI SATD_DETECTOR.PY ===\n')

# Test Comment
classify_satd('TODO: support multiple signers', source_type=DEF_COMMENT)
classify_satd('TODO: please add some javadoc', source_type=DEF_COMMENT)
classify_satd('TODO: lack of tests', source_type=DEF_COMMENT)

# Test Issue
classify_satd('I would like to remove this as its no longer needed.', source_type=DEF_ISSUE)
classify_satd('We need to update this documentation', source_type=DEF_ISSUE)
classify_satd('There are unimplemented requirements', source_type=DEF_ISSUE)
classify_satd('This is a good patch', source_type=DEF_ISSUE)

# Test Commit
classify_satd('fix bugs in SystemML - removed XXX', source_type=DEF_COMMIT)
classify_satd('fix typo in error message', source_type=DEF_COMMIT)

# Test Pull Request
classify_satd('nit: use local variable if possible', source_type=DEF_PULL)
classify_satd('Use the Python Postinstall implementation by default', source_type=DEF_PULL)